In [71]:
import os
import requests
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import geopandas as gpd

from google.cloud import bigquery
from dotenv import load_dotenv

In [71]:
load_dotenv()

SEOUL_API_KEY = os.getenv("SEOUL_API_KEY")

In [72]:
SEOUL_API_KEY is not None

True

# 데이터 불러오기

## 테스트

In [73]:
url = (
    f"http://openapi.seoul.go.kr:8088/"
    f"{SEOUL_API_KEY}/json/culturalEventInfo/1/5/"
)

response = requests.get(url)

response.status_code

200

In [74]:
data = response.json()

data.keys()

dict_keys(['culturalEventInfo'])

In [75]:
data["culturalEventInfo"].keys()

dict_keys(['list_total_count', 'RESULT', 'row'])

In [76]:
rows = data["culturalEventInfo"]["row"]

In [77]:
df = pd.DataFrame(rows)

df.head()

,CODENAME,GUNAME,TITLE,DATE,PLACE,ORG_NAME,USE_TRGT,USE_FEE,INQUIRY,PLAYER,...,RGSTDATE,TICKET,STRTDATE,END_DATE,THEMECODE,LOT,LAT,IS_FREE,HMPG_ADDR,PRO_TIME
0,콘서트,강동구,"2026 카즈미 타테이시 트리오 내한공연-크리스마스, 재즈를 만나다-(서울)",2026-12-24~2026-12-24,강동아트센터 대극장 한강,기타,"성인, 청소년","VIP석 88,000원 / R석 77,000원 / S석 66,000원 / A석 55...",070-8680-8477 / 02-337-3103,"Piano : Kazumi Tateishi, Contrabass : Shinobu ...",...,2026-07-23,시민,2026-12-24 00:00:00.0,2026-12-24 00:00:00.0,기타,127.157342546961,37.5512204558342,유료,https://culture.seoul.go.kr/culture/culture/cu...,19:30
1,콘서트,영등포구,"2026 카즈미 타테이시 트리오 내한공연-지브리, 재즈를 만나다-(서울)",2026-12-22~2026-12-22,영등포아트홀,기타,"성인, 청소년","VIP석 88,000원 / R석 77,000원 / S석 66,000원",070-8680-8477 / 02-337-3103,"Piano : Kazumi Tateishi, Contrabass : Shinobu ...",...,2026-07-16,시민,2026-12-22 00:00:00.0,2026-12-22 00:00:00.0,기타,126.900109255921,37.5260087284496,유료,https://culture.seoul.go.kr/culture/culture/cu...,19:30
2,연극,마포구,[마포문화재단] 체홉 4대 장막 낭독극 [공놀이클럽의 사계절 체홉: 갈매기],2026-11-29~2026-11-29,마포아트센터 아트홀맥,마포문화재단,14세 이상(2014년 이전 출생),전석 2만원(균일가),"02-3274-8600 [문의1번] 평일 9:00 ~ 18:00 (토,일 공휴일 휴무)",,...,2026-08-04,기관,2026-11-29 00:00:00.0,2026-11-29 00:00:00.0,기타,126.9455874749264,37.54987259578174,유료,https://culture.seoul.go.kr/culture/culture/cu...,(일) 16:00
3,전시/미술,중구,파인캐릭터 2026 (FineCharacter 2026),2026-11-27~2026-11-29,동대문디자인플라자(DDP) 쇼룸 1층 (서울 중구 을지로 281),기타,누구나,,031-921-6325,,...,2026-07-21,시민,2026-11-27 00:00:00.0,2026-11-29 00:00:00.0,기타,127.00977973484339,37.56735731522952,무료,https://culture.seoul.go.kr/culture/culture/cu...,10:00 ~ 19:00
4,콘서트,강북구,[꿈의숲아트센터] 꿈의숲 마티네 콘서트 [벨에포크 아트&뮤직] 시리즈3,2026-10-28~2026-10-28,북서울꿈의숲 상상톡톡미술관,세종문화회관,8세 이상 관람 가능,"전석 15,000원",02-399-1000,,...,2026-06-30,기관,2026-10-28 00:00:00.0,2026-10-28 00:00:00.0,기타,127.044324732036,37.6202544613023,유료,https://culture.seoul.go.kr/culture/culture/cu...,수요일 11:00


In [78]:
df.columns

Index(['CODENAME', 'GUNAME', 'TITLE', 'DATE', 'PLACE', 'ORG_NAME', 'USE_TRGT',
       'USE_FEE', 'INQUIRY', 'PLAYER', 'PROGRAM', 'ETC_DESC', 'ORG_LINK',
       'MAIN_IMG', 'RGSTDATE', 'TICKET', 'STRTDATE', 'END_DATE', 'THEMECODE',
       'LOT', 'LAT', 'IS_FREE', 'HMPG_ADDR', 'PRO_TIME'],
      dtype='str')

## 분석 데이터 불러오기

In [79]:
total_count = data["culturalEventInfo"]["list_total_count"]
all_rows = []

for start in range(1, total_count + 1, 1000):
    end = min(start + 999, total_count)

    url = (
        f"http://openapi.seoul.go.kr:8088/"
        f"{SEOUL_API_KEY}/json/culturalEventInfo/"
        f"{start}/{end}/"
    )

    response = requests.get(url)
    result = response.json()

    rows = result["culturalEventInfo"]["row"]
    all_rows.extend(rows)

In [80]:
df = pd.DataFrame(all_rows)
df.shape

(19502, 24)

In [81]:
print("API 전체 데이터 :", total_count)
print("실제 수집 데이터 :", len(df))
print("데이터 크기 :", df.shape)
print("중복 행 :", df.duplicated().sum())

API 전체 데이터 : 19502
실제 수집 데이터 : 19502
데이터 크기 : (19502, 24)
중복 행 : 0


# 데이터 전처리

## 불필요 컬럼 제거

In [82]:
df.head()

,CODENAME,GUNAME,TITLE,DATE,PLACE,ORG_NAME,USE_TRGT,USE_FEE,INQUIRY,PLAYER,...,RGSTDATE,TICKET,STRTDATE,END_DATE,THEMECODE,LOT,LAT,IS_FREE,HMPG_ADDR,PRO_TIME
0,콘서트,강동구,"2026 카즈미 타테이시 트리오 내한공연-크리스마스, 재즈를 만나다-(서울)",2026-12-24~2026-12-24,강동아트센터 대극장 한강,기타,"성인, 청소년","VIP석 88,000원 / R석 77,000원 / S석 66,000원 / A석 55...",070-8680-8477 / 02-337-3103,"Piano : Kazumi Tateishi, Contrabass : Shinobu ...",...,2026-07-23,시민,2026-12-24 00:00:00.0,2026-12-24 00:00:00.0,기타,127.157342546961,37.5512204558342,유료,https://culture.seoul.go.kr/culture/culture/cu...,19:30
1,콘서트,영등포구,"2026 카즈미 타테이시 트리오 내한공연-지브리, 재즈를 만나다-(서울)",2026-12-22~2026-12-22,영등포아트홀,기타,"성인, 청소년","VIP석 88,000원 / R석 77,000원 / S석 66,000원",070-8680-8477 / 02-337-3103,"Piano : Kazumi Tateishi, Contrabass : Shinobu ...",...,2026-07-16,시민,2026-12-22 00:00:00.0,2026-12-22 00:00:00.0,기타,126.900109255921,37.5260087284496,유료,https://culture.seoul.go.kr/culture/culture/cu...,19:30
2,연극,마포구,[마포문화재단] 체홉 4대 장막 낭독극 [공놀이클럽의 사계절 체홉: 갈매기],2026-11-29~2026-11-29,마포아트센터 아트홀맥,마포문화재단,14세 이상(2014년 이전 출생),전석 2만원(균일가),"02-3274-8600 [문의1번] 평일 9:00 ~ 18:00 (토,일 공휴일 휴무)",,...,2026-08-04,기관,2026-11-29 00:00:00.0,2026-11-29 00:00:00.0,기타,126.9455874749264,37.54987259578174,유료,https://culture.seoul.go.kr/culture/culture/cu...,(일) 16:00
3,전시/미술,중구,파인캐릭터 2026 (FineCharacter 2026),2026-11-27~2026-11-29,동대문디자인플라자(DDP) 쇼룸 1층 (서울 중구 을지로 281),기타,누구나,,031-921-6325,,...,2026-07-21,시민,2026-11-27 00:00:00.0,2026-11-29 00:00:00.0,기타,127.00977973484339,37.56735731522952,무료,https://culture.seoul.go.kr/culture/culture/cu...,10:00 ~ 19:00
4,콘서트,강북구,[꿈의숲아트센터] 꿈의숲 마티네 콘서트 [벨에포크 아트&뮤직] 시리즈3,2026-10-28~2026-10-28,북서울꿈의숲 상상톡톡미술관,세종문화회관,8세 이상 관람 가능,"전석 15,000원",02-399-1000,,...,2026-06-30,기관,2026-10-28 00:00:00.0,2026-10-28 00:00:00.0,기타,127.044324732036,37.6202544613023,유료,https://culture.seoul.go.kr/culture/culture/cu...,수요일 11:00


In [83]:
df.columns

Index(['CODENAME', 'GUNAME', 'TITLE', 'DATE', 'PLACE', 'ORG_NAME', 'USE_TRGT',
       'USE_FEE', 'INQUIRY', 'PLAYER', 'PROGRAM', 'ETC_DESC', 'ORG_LINK',
       'MAIN_IMG', 'RGSTDATE', 'TICKET', 'STRTDATE', 'END_DATE', 'THEMECODE',
       'LOT', 'LAT', 'IS_FREE', 'HMPG_ADDR', 'PRO_TIME'],
      dtype='str')

In [84]:
df = df[['CODENAME', 'GUNAME', 'TITLE', 'PLACE', 'ORG_NAME', 'USE_TRGT',
       'USE_FEE', 'RGSTDATE', 'TICKET', 'STRTDATE', 'END_DATE', 'THEMECODE',
       'LOT', 'LAT', 'IS_FREE', 'PRO_TIME', 'ORG_LINK', 'HMPG_ADDR']]

## 데이터 타입 변경

In [85]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 19502 entries, 0 to 19501
Data columns (total 18 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   CODENAME   19502 non-null  str  
 1   GUNAME     19502 non-null  str  
 2   TITLE      19502 non-null  str  
 3   PLACE      19502 non-null  str  
 4   ORG_NAME   19502 non-null  str  
 5   USE_TRGT   19502 non-null  str  
 6   USE_FEE    19502 non-null  str  
 7   RGSTDATE   19502 non-null  str  
 8   TICKET     19502 non-null  str  
 9   STRTDATE   19502 non-null  str  
 10  END_DATE   19502 non-null  str  
 11  THEMECODE  19502 non-null  str  
 12  LOT        19502 non-null  str  
 13  LAT        19502 non-null  str  
 14  IS_FREE    19502 non-null  str  
 15  PRO_TIME   19502 non-null  str  
 16  ORG_LINK   19502 non-null  str  
 17  HMPG_ADDR  19502 non-null  str  
dtypes: str(18)
memory usage: 2.7 MB


In [86]:
df['STRTDATE'] = pd.to_datetime(df['STRTDATE'])
df['END_DATE'] = pd.to_datetime(df['END_DATE'])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 19502 entries, 0 to 19501
Data columns (total 18 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   CODENAME   19502 non-null  str           
 1   GUNAME     19502 non-null  str           
 2   TITLE      19502 non-null  str           
 3   PLACE      19502 non-null  str           
 4   ORG_NAME   19502 non-null  str           
 5   USE_TRGT   19502 non-null  str           
 6   USE_FEE    19502 non-null  str           
 7   RGSTDATE   19502 non-null  str           
 8   TICKET     19502 non-null  str           
 9   STRTDATE   19502 non-null  datetime64[us]
 10  END_DATE   19502 non-null  datetime64[us]
 11  THEMECODE  19502 non-null  str           
 12  LOT        19502 non-null  str           
 13  LAT        19502 non-null  str           
 14  IS_FREE    19502 non-null  str           
 15  PRO_TIME   19502 non-null  str           
 16  ORG_LINK   19502 non-null  str           
 17  HMPG

In [87]:
df.head()

,CODENAME,GUNAME,TITLE,PLACE,ORG_NAME,USE_TRGT,USE_FEE,RGSTDATE,TICKET,STRTDATE,END_DATE,THEMECODE,LOT,LAT,IS_FREE,PRO_TIME,ORG_LINK,HMPG_ADDR
0,콘서트,강동구,"2026 카즈미 타테이시 트리오 내한공연-크리스마스, 재즈를 만나다-(서울)",강동아트센터 대극장 한강,기타,"성인, 청소년","VIP석 88,000원 / R석 77,000원 / S석 66,000원 / A석 55...",2026-07-23,시민,2026-12-24,2026-12-24,기타,127.157342546961,37.5512204558342,유료,19:30,https://tickets.interpark.com/goods/26010350,https://culture.seoul.go.kr/culture/culture/cu...
1,콘서트,영등포구,"2026 카즈미 타테이시 트리오 내한공연-지브리, 재즈를 만나다-(서울)",영등포아트홀,기타,"성인, 청소년","VIP석 88,000원 / R석 77,000원 / S석 66,000원",2026-07-16,시민,2026-12-22,2026-12-22,기타,126.900109255921,37.5260087284496,유료,19:30,https://tickets.interpark.com/goods/26010060,https://culture.seoul.go.kr/culture/culture/cu...
2,연극,마포구,[마포문화재단] 체홉 4대 장막 낭독극 [공놀이클럽의 사계절 체홉: 갈매기],마포아트센터 아트홀맥,마포문화재단,14세 이상(2014년 이전 출생),전석 2만원(균일가),2026-08-04,기관,2026-11-29,2026-11-29,기타,126.9455874749264,37.54987259578174,유료,(일) 16:00,https://www.mfac.or.kr/performance/whole_view....,https://culture.seoul.go.kr/culture/culture/cu...
3,전시/미술,중구,파인캐릭터 2026 (FineCharacter 2026),동대문디자인플라자(DDP) 쇼룸 1층 (서울 중구 을지로 281),기타,누구나,,2026-07-21,시민,2026-11-27,2026-11-29,기타,127.00977973484339,37.56735731522952,무료,10:00 ~ 19:00,https://finecharacter.kr/,https://culture.seoul.go.kr/culture/culture/cu...
4,콘서트,강북구,[꿈의숲아트센터] 꿈의숲 마티네 콘서트 [벨에포크 아트&뮤직] 시리즈3,북서울꿈의숲 상상톡톡미술관,세종문화회관,8세 이상 관람 가능,"전석 15,000원",2026-06-30,기관,2026-10-28,2026-10-28,기타,127.044324732036,37.6202544613023,유료,수요일 11:00,https://www.sejongpac.or.kr/dfac/dfacPerforman...,https://culture.seoul.go.kr/culture/culture/cu...


## 결측값 체크

In [88]:
df.isna().sum()

CODENAME     0
GUNAME       0
TITLE        0
PLACE        0
ORG_NAME     0
USE_TRGT     0
USE_FEE      0
RGSTDATE     0
TICKET       0
STRTDATE     0
END_DATE     0
THEMECODE    0
LOT          0
LAT          0
IS_FREE      0
PRO_TIME     0
ORG_LINK     0
HMPG_ADDR    0
dtype: int64

In [89]:
print("빈 문자열 개수")
print((df == "").sum())

빈 문자열 개수
CODENAME         0
GUNAME         124
TITLE            0
PLACE            0
ORG_NAME         0
USE_TRGT         0
USE_FEE      10501
RGSTDATE         0
TICKET           0
STRTDATE         0
END_DATE         0
THEMECODE     3329
LOT            521
LAT            521
IS_FREE          8
PRO_TIME         0
ORG_LINK       477
HMPG_ADDR        0
dtype: int64


In [90]:
len(df['USE_TRGT'].unique())

5008

In [91]:
df["USE_TRGT"].value_counts().head(30)

USE_TRGT
누구나                           4222
시민 누구나                         734
전체관람가                          677
성인                             482
홈페이지 참고                        469
초등학생 이상                        318
만 7세 이상                        302
8세 이상                          284
어린이                            166
초등학생 이상 관람가                    164
전체 관람가                         153
관심있는 누구나                       150
전체                             148
미취학아동 입장불가                     125
전 연령                           125
8세 이상 관람가                      107
36개월 이상                        104
프로그램별 상이                       104
7세 이상                           95
7세 이상 관람 가능 (2018년 이전 출생자)      89
모든 시민                           87
전연령                             82
5세 이상 어린이                       67
누구나                             66
7세 이상 관람 가능 (2019년 이전 출생자)      63
서울도서관 회원                        62
만 7세 이상                         59
일반시민                            58
초등학생       

In [92]:
df[df["USE_TRGT"] == "홈페이지 참고"]['HMPG_ADDR']

419      https://culture.seoul.go.kr/culture/culture/cu...
1101     https://culture.seoul.go.kr/culture/culture/cu...
2306     https://culture.seoul.go.kr/culture/culture/cu...
2515     https://culture.seoul.go.kr/culture/culture/cu...
3269     https://culture.seoul.go.kr/culture/culture/cu...
                               ...                        
18389    https://culture.seoul.go.kr/culture/culture/cu...
18400    https://culture.seoul.go.kr/culture/culture/cu...
18454    https://culture.seoul.go.kr/culture/culture/cu...
18573    https://culture.seoul.go.kr/culture/culture/cu...
18596    https://culture.seoul.go.kr/culture/culture/cu...
Name: HMPG_ADDR, Length: 469, dtype: str

행사 대상이 불명확한 데이터가 존재 함.


In [93]:
df["event_year"] = df["STRTDATE"].dt.year

df["event_year"].value_counts().sort_index()

event_year
2021     926
2022    3075
2023    3567
2024    5537
2025    3918
2026    2479
Name: count, dtype: int64

### 행사 시작일이 24년부터 현재까지의 데이터와 향후 예정인 데이터 대상으로 분석

In [94]:
df["STRTDATE"] = pd.to_datetime(
    df["STRTDATE"],
    errors="coerce"
)

start_date = pd.Timestamp("2024-01-01")

df_recent = df[
    df["STRTDATE"] >= start_date
].copy()

In [95]:
print("전체 행사 :", len(df))
print("최근 2년 행사 :", len(df_recent))

print(
    df_recent["STRTDATE"].min(),
    df_recent["STRTDATE"].max())

전체 행사 : 19502
최근 2년 행사 : 11934
2024-01-01 00:00:00 2026-12-24 00:00:00


# raw 데이터 내보내기

In [ ]:
df.to_csv(
    "../data/cultural_events_raw.csv",
    index=False
)

In [ ]:
df_recent.to_csv(
    "../data/cultural_events_2024_present.csv",
    index=False
)

In [100]:
df_c = pd.read_csv("C:\\workspace\\sprint_mission17\\data\\cultural_events_2024_present.csv")
df_c

,CODENAME,GUNAME,TITLE,PLACE,ORG_NAME,USE_TRGT,USE_FEE,RGSTDATE,TICKET,STRTDATE,END_DATE,THEMECODE,LOT,LAT,IS_FREE,PRO_TIME,ORG_LINK,HMPG_ADDR,event_year
0,콘서트,강동구,"2026 카즈미 타테이시 트리오 내한공연-크리스마스, 재즈를 만나다-(서울)",강동아트센터 대극장 한강,기타,"성인, 청소년","VIP석 88,000원 / R석 77,000원 / S석 66,000원 / A석 55...",2026-07-23,시민,2026-12-24,2026-12-24,기타,127.157342546961,37.5512204558342,유료,19:30,https://tickets.interpark.com/goods/26010350,https://culture.seoul.go.kr/culture/culture/cu...,2026
1,콘서트,영등포구,"2026 카즈미 타테이시 트리오 내한공연-지브리, 재즈를 만나다-(서울)",영등포아트홀,기타,"성인, 청소년","VIP석 88,000원 / R석 77,000원 / S석 66,000원",2026-07-16,시민,2026-12-22,2026-12-22,기타,126.900109255921,37.5260087284496,유료,19:30,https://tickets.interpark.com/goods/26010060,https://culture.seoul.go.kr/culture/culture/cu...,2026
2,연극,마포구,[마포문화재단] 체홉 4대 장막 낭독극 [공놀이클럽의 사계절 체홉: 갈매기],마포아트센터 아트홀맥,마포문화재단,14세 이상(2014년 이전 출생),전석 2만원(균일가),2026-08-04,기관,2026-11-29,2026-11-29,기타,126.9455874749264,37.54987259578174,유료,(일) 16:00,https://www.mfac.or.kr/performance/whole_view....,https://culture.seoul.go.kr/culture/culture/cu...,2026
3,전시/미술,중구,파인캐릭터 2026 (FineCharacter 2026),동대문디자인플라자(DDP) 쇼룸 1층 (서울 중구 을지로 281),기타,누구나,NaN,2026-07-21,시민,2026-11-27,2026-11-29,기타,127.00977973484339,37.56735731522952,무료,10:00 ~ 19:00,https://finecharacter.kr/,https://culture.seoul.go.kr/culture/culture/cu...,2026
4,콘서트,강북구,[꿈의숲아트센터] 꿈의숲 마티네 콘서트 [벨에포크 아트&뮤직] 시리즈3,북서울꿈의숲 상상톡톡미술관,세종문화회관,8세 이상 관람 가능,"전석 15,000원",2026-06-30,기관,2026-10-28,2026-10-28,기타,127.044324732036,37.6202544613023,유료,수요일 11:00,https://www.sejongpac.or.kr/dfac/dfacPerforman...,https://culture.seoul.go.kr/culture/culture/cu...,2026
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11929,교육/체험,중구,"[서울시립미술관] SeMA L [모으다-잇다, 마음구슬], [꼬리에 꼬리를 무는 미...",서울시립미술관 서소문본관 3층 프로젝트갤러리,서울시립미술관,누구나,NaN,2024-04-30,기관,2024-01-01,2024-12-31,기타,126.973699316136,37.5641060692766,무료,평일(화-금)오전 10시-오후 8시 토 · 일 · 공휴일 오전 10시-오후 7시 (...,https://sema.seoul.go.kr/kr/whatson/education/...,https://culture.seoul.go.kr/culture/culture/cu...,2024
11930,전시/미술,종로구,동대문 역사관,"2호선 동대문역사문화공원 1번, 2번 출구 / 4호선 동대문역 7번 출구",동대문디자인플라자,누구나,NaN,2024-01-02,기관,2024-01-01,2024-12-31,기타,127.01153416188667,37.56710262553757,무료,10:00 ~ 18:00 (※ 12~13시 휴관),https://ddp.or.kr/index.html?menuno=239&siteno...,https://culture.seoul.go.kr/culture/culture/cu...,2024
11931,전시/미술,중구,[DDP] 테테루뮤지엄 홀로그램 전시관,"DDP디자인랩 1층 C5호, C6호",동대문디자인플라자,전체관람가,"입장권 : 성인 8,000원, 유아동 및 초중고학생 5,000원",2024-10-16,기관,2024-01-01,2024-12-31,기타,127.00977973484339,37.56735731522952,유료,화-금(11:00-17:00) / 토-일(11:00-17:30) / 월요일휴관,https://ddp.or.kr/index.html?menuno=239&siteno...,https://culture.seoul.go.kr/culture/culture/cu...,2024
11932,교육/체험,송파구,2024 제1분기 송파문화원 수강생 모집,송파문화원,송파문화원,시민 누구나,프로그램별 상이,2023-12-29,기관,2024-01-01,2024-03-23,기타,127.075936810272,37.5106823860183,유료,프로그램별 상이,https://www.spcc.or.kr/spcccontents.asp?cc=050...,https://culture.seoul.go.kr/culture/culture/cu...,2024


In [102]:
df_c.isna().sum()

CODENAME         0
GUNAME          69
TITLE            0
PLACE            0
ORG_NAME         0
USE_TRGT         0
USE_FEE       6632
RGSTDATE         0
TICKET           0
STRTDATE         0
END_DATE         0
THEMECODE       65
LOT              1
LAT              1
IS_FREE          0
PRO_TIME         0
ORG_LINK         0
HMPG_ADDR        0
event_year       0
dtype: int64

# 2차 전처리

- 1차 전처리 후 당시 결측이 확인되지 않았으나, 내보낸 파일을 열었을 때 결측이 확인 됨에 따라 해당 결측처리를 진행함.
- 분석 및 대시보드 제작에 필요한 데이터 위주로 처리 (GUNAME, LOT, LAT)
- 좌표 데이터에 이상치 "-", "~" 데이터 확인 처리 필요.

## 결측 데이터 확인

In [103]:
df_c[df_c['GUNAME'].isna()]

,CODENAME,GUNAME,TITLE,PLACE,ORG_NAME,USE_TRGT,USE_FEE,RGSTDATE,TICKET,STRTDATE,END_DATE,THEMECODE,LOT,LAT,IS_FREE,PRO_TIME,ORG_LINK,HMPG_ADDR,event_year
197,전시/미술,NaN,"[서울시립미술관] 아마도, 모두 우리 [Perhaps, All of Us] 캐나다 전시",주캐나다한국문화원 1층 전시실,서울시립미술관,누구나,NaN,2026-08-10,기관,2026-08-25,2026-10-14,기타,75.6917°W,45.4215°N,무료,월 9:00~17:00 화-금 9:00~20:00 토 10:00~18:00 ※ 매주...,https://sema.seoul.go.kr/kr/whatson/exhibition...,https://culture.seoul.go.kr/culture/culture/cu...,2026
963,전시/미술,NaN,"[서울시립미술관] 2026 투어링 케이-아츠 《아마도, 모두 우리》","주워싱턴한국문화원 (2370 Massachusetts Ave., N.W. Washi...",서울시립미술관,누구나,NaN,2026-06-10,기관,2026-06-17,2026-08-11,기타,126.973699316136,37.5641060692766,무료,10:00 ~ 17:00 (12:00 ~ 13:00 관람 제한) 주말 및 미국 법정...,https://sema.seoul.go.kr/kr/whatson/exhibition...,https://culture.seoul.go.kr/culture/culture/cu...,2026
1840,축제-자연/경관,NaN,[서울대공원] 2026년 봄꽃축제,서울대공원,서울대공원,누구나,입장료 별도,2026-03-24,기관,2026-04-04,2026-04-12,기타,127.014098361931,37.4364305503019,유료,9:00 ~ 18:00,https://grandpark.seoul.go.kr/munhwa/munhwaVie...,https://culture.seoul.go.kr/culture/culture/cu...,2026
2453,교육/체험,NaN,[서울대공원] 겨울을 녹이는 남미관 이야기,서울동물원 남미관,서울대공원,"어린이, 청소년, 성인 등 누구나 참여가능",※ 입장료 별도,2026-01-28,기관,2026-01-05,2026-02-07,기타,127.014098361931,37.4364305503019,유료,1차 10:40~11:10 / 2차 13:20~13:50 / 3차 14:00~14:...,https://grandpark.seoul.go.kr/edu/program/view...,https://culture.seoul.go.kr/culture/culture/cu...,2026
3236,축제-문화/예술,NaN,[서울대공원] 2025 가을단풍축제 [가을빛 대공원],서울대공원 만남의 광장 일대,서울대공원,누구나,입장료 별도,2025-10-21,기관,2025-10-25,2025-11-02,기타,127.014098361931,37.4364305503019,유료,12:00 ~ 17:00,https://grandpark.seoul.go.kr/munhwa/munhwaVie...,https://culture.seoul.go.kr/culture/culture/cu...,2025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11443,교육/체험,NaN,[서울대공원] 물범 친구들을 소개합니다!,서울대공원 해양관,서울대공원,초등3~6학년,공원입장료,2024-02-27,기관,2024-03-09,2024-03-30,어린이/청소년 문화행사,127.014098361931,37.4364305503019,유료,13:20~15:00,https://grandpark.seoul.go.kr/edu/program/view...,https://culture.seoul.go.kr/culture/culture/cu...,2024
11444,교육/체험,NaN,[서울대공원] 물범 친구들을 소개합니다,서울동물원 해양관,서울대공원,어린이,동물원 입장료 별도,2024-02-08,기관,2024-03-09,2024-03-30,어린이/청소년 문화행사,127.014098361931,37.4364305503019,무료,매주 토 13:20~15:00,https://grandpark.seoul.go.kr/conts/contsView/...,https://culture.seoul.go.kr/culture/culture/cu...,2024
11662,교육/체험,NaN,[서울대공원] 2024 갑진년(甲辰年) 푸른 용과 멸종위기 동물 구해용!,서울대공원 동물원 제1아프리카관,서울대공원,어린이 등 관람객,NaN,2024-01-31,기관,2024-02-11,2024-02-18,어린이/청소년 문화행사,127.014098361931,37.4364305503019,무료,1차 11:00 2차 13:20 3차 14:00 4차 14:40,https://yeyak.seoul.go.kr/web/reservation/sele...,https://culture.seoul.go.kr/culture/culture/cu...,2024
11725,클래식,NaN,서울시립교향악단 특별 음악회 얍 판 츠베덴의 바그너 [발퀴레],세종예술의전당,서울시립교향악단,8세이상 관람가능,"VIP석 100,000원, R석 80,000원, S석 60,000원",2024-01-09,시민,2024-02-02,2024-02-02,기타,127.267380381461,36.4866171885619,유료,19:30,https://www.seoulphil.or.kr/perf/view?perfNo=5...,https://culture.seoul.go.kr/culture/culture/cu...,2024


In [104]:
missing_gu = df_c[
    df_c["GUNAME"].isna()
]

missing_gu[
    ["TITLE", "PLACE", "GUNAME", "LAT", "LOT"]
].head(20)

,TITLE,PLACE,GUNAME,LAT,LOT
197,"[서울시립미술관] 아마도, 모두 우리 [Perhaps, All of Us] 캐나다 전시",주캐나다한국문화원 1층 전시실,NaN,45.4215°N,75.6917°W
963,"[서울시립미술관] 2026 투어링 케이-아츠 《아마도, 모두 우리》","주워싱턴한국문화원 (2370 Massachusetts Ave., N.W. Washi...",NaN,37.5641060692766,126.973699316136
1840,[서울대공원] 2026년 봄꽃축제,서울대공원,NaN,37.4364305503019,127.014098361931
2453,[서울대공원] 겨울을 녹이는 남미관 이야기,서울동물원 남미관,NaN,37.4364305503019,127.014098361931
3236,[서울대공원] 2025 가을단풍축제 [가을빛 대공원],서울대공원 만남의 광장 일대,NaN,37.4364305503019,127.014098361931
4927,"[서울대공원] 2025 장미원·식물원 축제 [장미, 정원을 품다]","테마가든 장미원, 식물원",NaN,37.4364305503019,127.014098361931
5014,[광진문화재단] 2025 나루 동요제,서울어린이대공원 열린무대,NaN,37.54895853939875,127.077746180044
5078,[광진문화재단] 2025 피크닉 in 나루,서울어린이대공원 숲속의무대,NaN,37.54885117753382,127.07998387392475
5223,[서울대공원] 2025 어린이날 기념행사 [모여라! 대공원],서울대공원 동물원 일대,NaN,37.4364305503019,127.014098361931
5603,[서울대공원] 2025 서울대공원 벚꽃축제,서울대공원 일대,NaN,37.4364305503019,127.014098361931


In [105]:
missing_gu["PLACE"].value_counts()

PLACE
서울동물원                                                               6
서울대공원 산림치유센터                                                        6
서울대공원                                                               5
서울대공원 치유숲                                                           5
서울대공원 서울동물원                                                         3
서울대공원 테마가든                                                          3
서울대공원 일대                                                            2
각 기관                                                                2
서울대공원 식물원, 식물표본전시관                                                  2
서울대공원 호숫가 둘레길                                                       2
서울대공원 테마가든 내 어린이동물원                                                 2
온라인                                                                 2
서울대공원 식물원                                                           2
주캐나다한국문화원 1층 전시실                                                    1
주워싱턴한국문화원 (237

### 동일 좌표의 정보로 결측값 대체

In [106]:
coord_to_gu = (
    df_c
    .dropna(subset=["GUNAME", "LAT", "LOT"])
    .drop_duplicates(subset=["LAT", "LOT"])
    .set_index(["LAT", "LOT"])["GUNAME"]
    .to_dict()
)

In [107]:
mask = df_c["GUNAME"].isna()

df_c.loc[mask, "GUNAME"] = (
    df_c.loc[mask]
    .apply(
        lambda x: coord_to_gu.get((x["LAT"], x["LOT"])),
        axis=1
    )
)

In [108]:
df_c["GUNAME"].isna().sum()

np.int64(3)

In [109]:
df_c[df_c["GUNAME"].isna()]

,CODENAME,GUNAME,TITLE,PLACE,ORG_NAME,USE_TRGT,USE_FEE,RGSTDATE,TICKET,STRTDATE,END_DATE,THEMECODE,LOT,LAT,IS_FREE,PRO_TIME,ORG_LINK,HMPG_ADDR,event_year
197,전시/미술,NaN,"[서울시립미술관] 아마도, 모두 우리 [Perhaps, All of Us] 캐나다 전시",주캐나다한국문화원 1층 전시실,서울시립미술관,누구나,NaN,2026-08-10,기관,2026-08-25,2026-10-14,기타,75.6917°W,45.4215°N,무료,월 9:00~17:00 화-금 9:00~20:00 토 10:00~18:00 ※ 매주...,https://sema.seoul.go.kr/kr/whatson/exhibition...,https://culture.seoul.go.kr/culture/culture/cu...,2026
9342,축제-기타,NaN,2024 한강페스티벌 여름축제 [한강음악불꽃크루즈],아라김포여객터미널,한강사업본부,누구나,"대인 40,000원, 소인 25,000원",2024-07-02,기관,2024-07-27,2024-08-10,기타,126.786970046495,37.5976552679624,유료,매주 토요일 18:00~ (18:30 출항) (8.4.(일) 추가),https://hangang.seoul.go.kr/www/eventMng/detai...,https://culture.seoul.go.kr/culture/culture/cu...,2024
11725,클래식,NaN,서울시립교향악단 특별 음악회 얍 판 츠베덴의 바그너 [발퀴레],세종예술의전당,서울시립교향악단,8세이상 관람가능,"VIP석 100,000원, R석 80,000원, S석 60,000원",2024-01-09,시민,2024-02-02,2024-02-02,기타,127.267380381461,36.4866171885619,유료,19:30,https://www.seoulphil.or.kr/perf/view?perfNo=5...,https://culture.seoul.go.kr/culture/culture/cu...,2024


In [68]:
df_c.shape

(11934, 19)

In [110]:
df_c = df_c.dropna(subset='GUNAME')
df_c.shape

(11931, 19)

### 서울이 아닌 위치 확인 후 3개의 행사 제거

- 결측이 채워지지않는 좌표의 행사를 조회하여 위치를 파악 후 서울이 아닌 관계로 제거.

In [112]:
df_c[df_c['LOT'].isna()]

,CODENAME,GUNAME,TITLE,PLACE,ORG_NAME,USE_TRGT,USE_FEE,RGSTDATE,TICKET,STRTDATE,END_DATE,THEMECODE,LOT,LAT,IS_FREE,PRO_TIME,ORG_LINK,HMPG_ADDR,event_year
4902,교육/체험,강서구,리드인 X 고명환 작가와 함께하는 [인문학 콘서트],강서시니어스타운 B1 송도아트홀,기타,독서논술 리드인 학부모님 또는 리드인 교육에 관심있는 분 누구나,무료,2025-04-24,시민,2025-06-03,2025-06-03,기타,NaN,NaN,무료,10:30~행사 종료시까지,https://blog.naver.com/readin6/223858472466,https://culture.seoul.go.kr/culture/culture/cu...,2025


In [113]:
df_c[df_c["PLACE"] == "강서시니어스타운 B1 송도아트홀"]

,CODENAME,GUNAME,TITLE,PLACE,ORG_NAME,USE_TRGT,USE_FEE,RGSTDATE,TICKET,STRTDATE,END_DATE,THEMECODE,LOT,LAT,IS_FREE,PRO_TIME,ORG_LINK,HMPG_ADDR,event_year
4902,교육/체험,강서구,리드인 X 고명환 작가와 함께하는 [인문학 콘서트],강서시니어스타운 B1 송도아트홀,기타,독서논술 리드인 학부모님 또는 리드인 교육에 관심있는 분 누구나,무료,2025-04-24,시민,2025-06-03,2025-06-03,기타,NaN,NaN,무료,10:30~행사 종료시까지,https://blog.naver.com/readin6/223858472466,https://culture.seoul.go.kr/culture/culture/cu...,2025


In [114]:
df_c['LOT'] = df_c['LOT'].fillna(126.85943092649)
df_c['LAT'] = df_c['LAT'].fillna(37.557203266398)
df_c[df_c["PLACE"] == "강서시니어스타운 B1 송도아트홀"]

,CODENAME,GUNAME,TITLE,PLACE,ORG_NAME,USE_TRGT,USE_FEE,RGSTDATE,TICKET,STRTDATE,END_DATE,THEMECODE,LOT,LAT,IS_FREE,PRO_TIME,ORG_LINK,HMPG_ADDR,event_year
4902,교육/체험,강서구,리드인 X 고명환 작가와 함께하는 [인문학 콘서트],강서시니어스타운 B1 송도아트홀,기타,독서논술 리드인 학부모님 또는 리드인 교육에 관심있는 분 누구나,무료,2025-04-24,시민,2025-06-03,2025-06-03,기타,126.859431,37.557203,무료,10:30~행사 종료시까지,https://blog.naver.com/readin6/223858472466,https://culture.seoul.go.kr/culture/culture/cu...,2025


### 강서구 좌표 결측을 장소를 조회하여 값을 대체

- 좌표가 결측인 값 1건을 검색하여 좌표 입력.

## 이상치 데이터 처리

In [117]:
df_c[df_c['PLACE'] == "갤러리 마리"]

,CODENAME,GUNAME,TITLE,PLACE,ORG_NAME,USE_TRGT,USE_FEE,RGSTDATE,TICKET,STRTDATE,END_DATE,THEMECODE,LOT,LAT,IS_FREE,PRO_TIME,ORG_LINK,HMPG_ADDR,event_year
6224,전시/미술,종로구,푸른 뱀띠 해 특별전 [을사 1905-2025 : A New Dream in the...,갤러리 마리,기타,누구나,NaN,2025-02-14,시민,2025-01-17,2025-02-28,기타,-,-,무료,화-토 11:00~19:00 / 일-월 휴무,http://www.gallerymarie.org/korean/viewforum.p...,https://culture.seoul.go.kr/culture/culture/cu...,2025
7144,전시/미술,종로구,Beyond Genre [장르탈출 Ⅱ],갤러리 마리,기타,누구나,NaN,2024-12-14,시민,2024-11-22,2025-01-10,기타,-,-,무료,"화-토 11:00~19:00 / 일-월요일, 신정 1월 1일 휴무",http://www.gallerymarie.org/korean/viewtopic.p...,https://culture.seoul.go.kr/culture/culture/cu...,2024
7956,전시/미술,종로구,"[추니박 개인전] 가보지 않은 길, 낯선 풍경",갤러리 마리,기타,누구나,NaN,2024-10-18,시민,2024-10-11,2024-11-15,기타,-,-,무료,화-토 11:00~19:00 / 일-월 휴무,http://www.gallerymarie.org/korean/viewtopic.p...,https://culture.seoul.go.kr/culture/culture/cu...,2024
8757,전시/미술,종로구,"윤형선 [Dance of Flowers, Voice of Nature - 춤추는 꽃]",갤러리 마리,기타,누구나,NaN,2024-09-06,시민,2024-08-30,2024-10-04,기타,-,-,무료,"화~토 11:00~19:00 / 매주 일~월, 추석연휴 휴무",http://www.gallerymarie.org/korean/viewforum.p...,https://culture.seoul.go.kr/culture/culture/cu...,2024


In [118]:
df_c[df_c['LOT'] == "-"]

,CODENAME,GUNAME,TITLE,PLACE,ORG_NAME,USE_TRGT,USE_FEE,RGSTDATE,TICKET,STRTDATE,END_DATE,THEMECODE,LOT,LAT,IS_FREE,PRO_TIME,ORG_LINK,HMPG_ADDR,event_year
6224,전시/미술,종로구,푸른 뱀띠 해 특별전 [을사 1905-2025 : A New Dream in the...,갤러리 마리,기타,누구나,NaN,2025-02-14,시민,2025-01-17,2025-02-28,기타,-,-,무료,화-토 11:00~19:00 / 일-월 휴무,http://www.gallerymarie.org/korean/viewforum.p...,https://culture.seoul.go.kr/culture/culture/cu...,2025
7144,전시/미술,종로구,Beyond Genre [장르탈출 Ⅱ],갤러리 마리,기타,누구나,NaN,2024-12-14,시민,2024-11-22,2025-01-10,기타,-,-,무료,"화-토 11:00~19:00 / 일-월요일, 신정 1월 1일 휴무",http://www.gallerymarie.org/korean/viewtopic.p...,https://culture.seoul.go.kr/culture/culture/cu...,2024
7956,전시/미술,종로구,"[추니박 개인전] 가보지 않은 길, 낯선 풍경",갤러리 마리,기타,누구나,NaN,2024-10-18,시민,2024-10-11,2024-11-15,기타,-,-,무료,화-토 11:00~19:00 / 일-월 휴무,http://www.gallerymarie.org/korean/viewtopic.p...,https://culture.seoul.go.kr/culture/culture/cu...,2024
8757,전시/미술,종로구,"윤형선 [Dance of Flowers, Voice of Nature - 춤추는 꽃]",갤러리 마리,기타,누구나,NaN,2024-09-06,시민,2024-08-30,2024-10-04,기타,-,-,무료,"화~토 11:00~19:00 / 매주 일~월, 추석연휴 휴무",http://www.gallerymarie.org/korean/viewforum.p...,https://culture.seoul.go.kr/culture/culture/cu...,2024
9399,전시/미술,종로구,한불조형예술협회 [산책-PROMENADE],갤러리 마리,기타,누구나,NaN,2024-08-01,시민,2024-07-25,2024-08-16,기타,-,-,무료,화~토 11:00~19:00 / 매주 일~월 휴무,https://www.instagram.com/p/C9wlMzRS_bz/,https://culture.seoul.go.kr/culture/culture/cu...,2024


In [119]:
lot_num = pd.to_numeric(df_c["LOT"], errors="coerce")
lat_num = pd.to_numeric(df_c["LAT"], errors="coerce")

invalid_lot = df_c[
    df_c["LOT"].notna() & lot_num.isna()
]

invalid_lat = df_c[
    df_c["LAT"].notna() & lat_num.isna()
]


In [120]:
invalid_lot["LOT"].value_counts()

LOT
-    5
Name: count, dtype: int64

In [121]:
invalid_lat["LAT"].value_counts()

LAT
-                     5
37.5718961547884~2    1
Name: count, dtype: int64

In [122]:
df_c[df_c['LAT'] == '37.5718961547884~2']

,CODENAME,GUNAME,TITLE,PLACE,ORG_NAME,USE_TRGT,USE_FEE,RGSTDATE,TICKET,STRTDATE,END_DATE,THEMECODE,LOT,LAT,IS_FREE,PRO_TIME,ORG_LINK,HMPG_ADDR,event_year
5195,축제-문화/예술,종로구,2025 국제기로 미술대축제,인사동 한국미술관 2층 전시관,기타,누구나,NaN,2025-04-21,시민,2025-05-07,2025-05-12,기타,126.987230558854,37.5718961547884~2,무료,10:00~18:00,https://www.kiroart.or.kr/,https://culture.seoul.go.kr/culture/culture/cu...,2025


In [ ]:
df_c["LAT"] = df_c["LAT"].replace("37.5718961547884~2", 37.57189615478842)
df_c["LAT"] = df_c["LAT"].replace("-", 37.5727115)
df_c["LOT"] = df_c["LOT"].replace("-", 126.9690368)


,CODENAME,GUNAME,TITLE,PLACE,ORG_NAME,USE_TRGT,USE_FEE,RGSTDATE,TICKET,STRTDATE,END_DATE,THEMECODE,LOT,LAT,IS_FREE,PRO_TIME,ORG_LINK,HMPG_ADDR,event_year


In [126]:
df_c[df_c["PLACE"] == '갤러리 마리']

,CODENAME,GUNAME,TITLE,PLACE,ORG_NAME,USE_TRGT,USE_FEE,RGSTDATE,TICKET,STRTDATE,END_DATE,THEMECODE,LOT,LAT,IS_FREE,PRO_TIME,ORG_LINK,HMPG_ADDR,event_year
6224,전시/미술,종로구,푸른 뱀띠 해 특별전 [을사 1905-2025 : A New Dream in the...,갤러리 마리,기타,누구나,NaN,2025-02-14,시민,2025-01-17,2025-02-28,기타,126.969037,37.572711,무료,화-토 11:00~19:00 / 일-월 휴무,http://www.gallerymarie.org/korean/viewforum.p...,https://culture.seoul.go.kr/culture/culture/cu...,2025
7144,전시/미술,종로구,Beyond Genre [장르탈출 Ⅱ],갤러리 마리,기타,누구나,NaN,2024-12-14,시민,2024-11-22,2025-01-10,기타,126.969037,37.572711,무료,"화-토 11:00~19:00 / 일-월요일, 신정 1월 1일 휴무",http://www.gallerymarie.org/korean/viewtopic.p...,https://culture.seoul.go.kr/culture/culture/cu...,2024
7956,전시/미술,종로구,"[추니박 개인전] 가보지 않은 길, 낯선 풍경",갤러리 마리,기타,누구나,NaN,2024-10-18,시민,2024-10-11,2024-11-15,기타,126.969037,37.572711,무료,화-토 11:00~19:00 / 일-월 휴무,http://www.gallerymarie.org/korean/viewtopic.p...,https://culture.seoul.go.kr/culture/culture/cu...,2024
8757,전시/미술,종로구,"윤형선 [Dance of Flowers, Voice of Nature - 춤추는 꽃]",갤러리 마리,기타,누구나,NaN,2024-09-06,시민,2024-08-30,2024-10-04,기타,126.969037,37.572711,무료,"화~토 11:00~19:00 / 매주 일~월, 추석연휴 휴무",http://www.gallerymarie.org/korean/viewforum.p...,https://culture.seoul.go.kr/culture/culture/cu...,2024


In [127]:
df_c.to_csv("../data/cultural_events_2024_present.csv",
    index=False)

In [129]:
df_c["THEMECODE"].unique()

<StringArray>
['기타', '어린이/청소년 문화행사', '어르신 문화행사', '가족 문화행사', nan, '여성 문화행사', '문화가 있는 날']
Length: 7, dtype: str

# 인구 데이터 확인

In [ ]:
years = [2024, 2025, 2026]

population_dfs = {}

for year in years:
    
    # 1. CSV 불러오기
    df_1 = pd.read_csv(
        f"../data/등록인구(연령별_동별)_{year}.csv"
    )

    # 2. 0번 행을 컬럼명으로 지정
    df_1.columns = df_1.iloc[0]

    # 3. 기존 0번 행 제거
    df_1 = df_1.iloc[1:].reset_index(drop=True)

    # 4. 전체 합계 제외 + 한국인만 선택
    df_1 = df_1[
        (df_1["동별(1)"] != "합계") &
        (df_1["항목"] == "한국인")
    ].copy()

    # 5. 딕셔너리에 저장
    population_dfs[year] = df_1

In [137]:
rename_columns = {
    "동별(1)": "gu_name",
    "항목": "population_type",
    "합계": "total_population",
    "0~4세": "age_0_4",
    "5~9세": "age_5_9",
    "10~14세": "age_10_14",
    "15~19세": "age_15_19",
    "20~24세": "age_20_24",
    "25~29세": "age_25_29",
    "30~34세": "age_30_34",
    "35~39세": "age_35_39",
    "40~44세": "age_40_44",
    "45~49세": "age_45_49",
    "50~54세": "age_50_54",
    "55~59세": "age_55_59",
    "60~64세": "age_60_64",
    "65~69세": "age_65_69",
    "70~74세": "age_70_74",
    "75~79세": "age_75_79",
    "80~84세": "age_80_84",
    "85~89세": "age_85_89",
    "90~94세": "age_90_94",
    "95~99세": "age_95_99",
    "100세 이상": "age_100_plus",
    "연도": "year"
}

for year, df in population_dfs.items():
    df = df.rename(columns=rename_columns)
    df = df.drop(columns=["population_type"])
    
    population_dfs[year] = df

In [138]:
for year, df in population_dfs.items():
    df.to_csv(
        f"../data/population_{year}.csv",
        index=False,
        encoding="utf-8-sig"
    )

# bigquery에서 데이터 불러오기

In [2]:
client = bigquery.Client(
    project="sprint-mission17"
)

In [3]:
query = """
SELECT *
FROM `sprint-mission17.mission17_kkw.cultural`
"""

df_cultural = client.query(query).to_dataframe()

query2024 = """
SELECT *
FROM `sprint-mission17.mission17_kkw.p_2024`
"""

df_p2024 = client.query(query2024).to_dataframe()

query2025 = """
SELECT *
FROM `sprint-mission17.mission17_kkw.p_2025`
"""

df_p2025 = client.query(query2025).to_dataframe()

query2026 = """
SELECT *
FROM `sprint-mission17.mission17_kkw.p_2026`
"""

df_p2026 = client.query(query2026).to_dataframe()

c:\workspace\sprint_mission17\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
c:\workspace\sprint_mission17\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
c:\workspace\sprint_mission17\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
c:\workspace\sprint_mission17\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [4]:
df_cultural.head()

,CODENAME,GUNAME,TITLE,PLACE,ORG_NAME,USE_TRGT,USE_FEE,RGSTDATE,TICKET,STRTDATE,END_DATE,THEMECODE,LOT,LAT,IS_FREE,PRO_TIME,ORG_LINK,HMPG_ADDR,event_year
0,교육/체험,광진구,[서울대공원] 2025 띵똥! 서울동물원 교육꾸러미 비대면 교육 (초등학교 특수학급...,각 기관,서울대공원,초등학교 특수학급,NaN,2025-03-11,기관,2025-04-01,2025-10-31,어린이/청소년 문화행사,127.014098,37.436431,무료,각 기관 별 상이,https://grandpark.seoul.go.kr/edu/program/view...,https://culture.seoul.go.kr/culture/culture/cu...,2025
1,교육/체험,광진구,[서울대공원] 2025 [학교에서 만나는 동물원] 비대면 교육,각 기관,서울대공원,초등학교 특수학급,NaN,2025-03-11,기관,2025-04-01,2025-10-31,어린이/청소년 문화행사,127.014098,37.436431,무료,각 기관 별 상이,https://grandpark.seoul.go.kr/edu/program/view...,https://culture.seoul.go.kr/culture/culture/cu...,2025
2,교육/체험,광진구,[서울대공원] 2024 생물다양성의 날 기념 멸종위기종 보전 교육,동물원 내 100주년 기념 광장,서울대공원,동물원 관람객,홈페이지 참고,2024-05-21,기관,2024-05-22,2024-05-26,기타,127.014098,37.436431,유료,"10:30~11:30, 13:20~15:30",https://grandpark.seoul.go.kr/board/view/ko/S0...,https://culture.seoul.go.kr/culture/culture/cu...,2024
3,교육/체험,광진구,[서울대공원] 사자야! 신나게 놀아보자!,"동물원 내 식물원(세미나실), 사자 실내 관람장",서울대공원,초등학교 1~3학년,NaN,2024-07-23,기관,2024-07-31,2024-08-14,어린이/청소년 문화행사,127.014098,37.436431,무료,13:40-14:40,https://grandpark.seoul.go.kr/edu/program/view...,https://culture.seoul.go.kr/culture/culture/cu...,2024
4,축제-자연/경관,광진구,[서울대공원] 2026년 봄꽃축제,서울대공원,서울대공원,누구나,입장료 별도,2026-03-24,기관,2026-04-04,2026-04-12,기타,127.014098,37.436431,유료,9:00 ~ 18:00,https://grandpark.seoul.go.kr/munhwa/munhwaVie...,https://culture.seoul.go.kr/culture/culture/cu...,2026


In [5]:
total_cul = df_cultural.groupby(['event_year', 'GUNAME'])['TITLE'].count().reset_index()
total_cul.head()


,event_year,GUNAME,TITLE
0,2024,강남구,216
1,2024,강동구,83
2,2024,강북구,250
3,2024,강서구,84
4,2024,관악구,102


In [6]:
df_p2024['year'] = 2024
df_p2025['year'] = 2025
df_p2026['year'] = 2026

In [7]:
p_data = pd.concat(
    [df_p2024, df_p2025, df_p2026],
    ignore_index=True
)
p_data

,gu_name,total_population,age_0_4,age_5_9,age_10_14,age_15_19,age_20_24,age_25_29,age_30_34,age_35_39,...,age_60_64,age_65_69,age_70_74,age_75_79,age_80_84,age_85_89,age_90_94,age_95_99,age_100_plus,year
0,중구,120544,2562,2754,2803,3080,6346,10431,12014,9582,...,9568,8841,6021,4824,3747,2027,728,198,22,2024
1,성동구,273669,7563,8411,8237,9094,16106,22847,24802,21622,...,20751,17867,11745,9348,7001,3634,1156,245,34,2024
2,금천구,225159,4099,4927,5885,6540,10779,22791,23016,15502,...,19847,17033,11508,8644,5890,2860,899,168,35,2024
3,종로구,138336,2254,3257,4420,4958,8639,12324,11326,8428,...,11293,9723,6478,5389,4337,2500,884,222,41,2024
4,강북구,283597,4189,6365,8573,9832,15996,20206,19307,15295,...,24927,23150,16766,13953,10529,5030,1405,350,44,2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,강동구,499011,14049,17504,22642,21374,20888,33293,40323,36893,...,34819,37085,27574,17866,10515,5157,2024,512,80,2026
71,노원구,481053,10084,12346,19775,23739,28632,31302,32985,28859,...,35723,36537,27187,18828,13211,7659,2932,656,85,2026
72,송파구,650258,16945,22553,29320,28530,28509,46411,56419,50409,...,43897,44554,33458,22131,13080,6680,2696,640,88,2026
73,강남구,552830,12515,18736,31648,31977,25762,35697,41341,37927,...,31848,31511,26418,19031,11649,6028,2272,587,92,2026


In [8]:
total_cul = pd.merge(total_cul, p_data, left_on=["event_year", "GUNAME"], right_on=["year", "gu_name"], how='left')
total_cul

,event_year,GUNAME,TITLE,gu_name,total_population,age_0_4,age_5_9,age_10_14,age_15_19,age_20_24,...,age_60_64,age_65_69,age_70_74,age_75_79,age_80_84,age_85_89,age_90_94,age_95_99,age_100_plus,year
0,2024,강남구,216,강남구,557345,12273,20887,32396,31131,28418,...,32965,31274,23259,18050,10828,5271,1970,556,83,2024
1,2024,강동구,83,강동구,476942,13137,17932,20943,19891,22583,...,36512,34129,23008,16145,9632,4557,1668,467,81,2024
2,2024,강북구,250,강북구,283597,4189,6365,8573,9832,15996,...,24927,23150,16766,13953,10529,5030,1405,350,44,2024
3,2024,강서구,84,강서구,556171,12403,16457,20183,20576,27891,...,41148,39276,26829,19388,12318,6439,2330,580,86,2024
4,2024,관악구,102,관악구,477812,6591,8432,10778,13318,34048,...,31464,29105,21507,17469,12378,5646,1627,356,50,2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,2026,용산구,72,용산구,199648,4747,4990,5993,6576,9480,...,14368,13480,9630,6794,5125,3368,1358,307,58,2026
71,2026,은평구,148,은평구,453212,9157,10627,15338,17741,21278,...,34123,34995,25838,19281,13772,7776,2604,553,74,2026
72,2026,종로구,618,종로구,136318,2207,2766,4051,5011,7907,...,10703,10220,7482,5349,4228,2668,1026,250,40,2026
73,2026,중구,262,중구,117558,2487,2341,2673,3168,5666,...,8730,9030,6839,4925,3648,2194,827,212,26,2026


In [9]:
total_cul['grade_cult'] = (total_cul['TITLE'] / total_cul['total_population']) * 10000
total_cul.head()

,event_year,GUNAME,TITLE,gu_name,total_population,age_0_4,age_5_9,age_10_14,age_15_19,age_20_24,...,age_65_69,age_70_74,age_75_79,age_80_84,age_85_89,age_90_94,age_95_99,age_100_plus,year,grade_cult
0,2024,강남구,216,강남구,557345,12273,20887,32396,31131,28418,...,31274,23259,18050,10828,5271,1970,556,83,2024,3.875517
1,2024,강동구,83,강동구,476942,13137,17932,20943,19891,22583,...,34129,23008,16145,9632,4557,1668,467,81,2024,1.740254
2,2024,강북구,250,강북구,283597,4189,6365,8573,9832,15996,...,23150,16766,13953,10529,5030,1405,350,44,2024,8.815326
3,2024,강서구,84,강서구,556171,12403,16457,20183,20576,27891,...,39276,26829,19388,12318,6439,2330,580,86,2024,1.510327
4,2024,관악구,102,관악구,477812,6591,8432,10778,13318,34048,...,29105,21507,17469,12378,5646,1627,356,50,2024,2.134731


In [10]:
df_cultural["USE_TRGT"].value_counts().reset_index().sort_values(by='count', ascending=False).head(30)


,USE_TRGT,count
0,누구나,3226
1,전체관람가,384
2,시민 누구나,361
3,성인,354
4,초등학생 이상,282
5,초등학생 이상 관람가,147
6,어린이,138
7,관심있는 누구나,131
8,전체 관람가,105
9,만 7세 이상,94


In [11]:
df_cultural['THEMECODE'].value_counts()

THEMECODE
기타              9841
어린이/청소년 문화행사    1477
가족 문화행사          399
어르신 문화행사          94
문화가 있는 날          30
여성 문화행사           25
Name: count, dtype: int64

In [12]:
df_cultural[df_cultural['THEMECODE'] == '기타'].head(20)

,CODENAME,GUNAME,TITLE,PLACE,ORG_NAME,USE_TRGT,USE_FEE,RGSTDATE,TICKET,STRTDATE,END_DATE,THEMECODE,LOT,LAT,IS_FREE,PRO_TIME,ORG_LINK,HMPG_ADDR,event_year
2,교육/체험,광진구,[서울대공원] 2024 생물다양성의 날 기념 멸종위기종 보전 교육,동물원 내 100주년 기념 광장,서울대공원,동물원 관람객,홈페이지 참고,2024-05-21,기관,2024-05-22,2024-05-26,기타,127.014098,37.436431,유료,"10:30~11:30, 13:20~15:30",https://grandpark.seoul.go.kr/board/view/ko/S0...,https://culture.seoul.go.kr/culture/culture/cu...,2024
4,축제-자연/경관,광진구,[서울대공원] 2026년 봄꽃축제,서울대공원,서울대공원,누구나,입장료 별도,2026-03-24,기관,2026-04-04,2026-04-12,기타,127.014098,37.436431,유료,9:00 ~ 18:00,https://grandpark.seoul.go.kr/munhwa/munhwaVie...,https://culture.seoul.go.kr/culture/culture/cu...,2026
9,교육/체험,광진구,[서울대공원] 산림치유센터_활력드림,서울대공원 내 산림치유센터,서울대공원,"단체, 동호회 / 회차별 10명",NaN,2024-11-07,기관,2024-09-01,2024-11-30,기타,127.014098,37.436431,무료,매주 금 13:00-14:30,https://grandpark.seoul.go.kr/edu/program/view...,https://culture.seoul.go.kr/culture/culture/cu...,2024
12,교육/체험,광진구,[서울대공원] 동양관 속 멸종위기 동물 이야기,서울대공원 동양관 실내,서울대공원,"어린이, 청소년, 학부모 등",동물원 입장료 별도,2025-03-07,기관,2025-03-08,2025-03-29,기타,127.014098,37.436431,무료,1차 10:40~11:20/ 2차 13:20~14:00/ 3차 14:20~15:00,https://grandpark.seoul.go.kr/edu/program/view...,https://culture.seoul.go.kr/culture/culture/cu...,2025
14,축제-문화/예술,광진구,[서울대공원] 2025 가을단풍축제 [가을빛 대공원],서울대공원 만남의 광장 일대,서울대공원,누구나,입장료 별도,2025-10-21,기관,2025-10-25,2025-11-02,기타,127.014098,37.436431,유료,12:00 ~ 17:00,https://grandpark.seoul.go.kr/munhwa/munhwaVie...,https://culture.seoul.go.kr/culture/culture/cu...,2025
15,전시/미술,광진구,[서울대공원] 2024년 야외전시 [동물원 밖 동물원],서울대공원 만남의 광장 일원,서울대공원,누구나,무료,2024-10-24,기관,2024-10-26,2024-11-16,기타,127.014098,37.436431,무료,기간 중 상시,https://grandpark.seoul.go.kr/board/view/ko/S0...,https://culture.seoul.go.kr/culture/culture/cu...,2024
16,교육/체험,광진구,[서울대공원] 산림치유센터_여유드림 9월,서울대공원 산림치유센터,서울대공원,일반(성인),무료,2024-08-12,기관,2024-09-02,2024-11-30,기타,127.014098,37.436431,무료,상세내용 참고,https://grandpark.seoul.go.kr/edu/program/view...,https://culture.seoul.go.kr/culture/culture/cu...,2024
17,교육/체험,광진구,[서울대공원] 행복드림_9월,서울대공원 산림치유센터,서울대공원,단체(사회적배려층 단체) / 돌봄자 포함 회당 15명(최소 5명~최대 15명),무료,2024-08-12,기관,2024-09-02,2024-09-30,기타,127.014098,37.436431,무료,화 13:00~14:30,https://grandpark.seoul.go.kr/edu/program/view...,https://culture.seoul.go.kr/culture/culture/cu...,2024
18,교육/체험,광진구,[서울대공원] 2024 활력드림_9월,서울대공원 산림치유센터,서울대공원,"단체, 동호회 / 회차별 10명",무료,2024-08-12,기관,2024-09-02,2024-09-30,기타,127.014098,37.436431,무료,금 13:00~14:30,https://yeyak.seoul.go.kr/web/reservation/sele...,https://culture.seoul.go.kr/culture/culture/cu...,2024
19,교육/체험,광진구,[서울대공원] 산림치유센터_여유드림,서울대공원 산림치유센터,서울대공원,성인,무료,2024-05-06,기관,2024-06-03,2024-06-28,기타,127.014098,37.436431,무료,"월, 목, 금: 9:30~11:00 / 화, 수, 금: 13:00~14:00 / 월...",https://grandpark.seoul.go.kr/edu/program/view...,https://culture.seoul.go.kr/culture/culture/cu...,2024


# 대상 정규화

In [13]:
def classify_scope(text):
    if pd.isna(text):
        return "기타"

    text = str(text).strip()

    # 지역 제한
    if re.search(r"구민|주민|서울시민|시민 대상", text):
        return "지역제한"

    # 회원 또는 특정 대상
    if re.search(r"회원|이용자|수강생|참가자", text):
        return "회원·특정대상"

    # 연령 제한
    if re.search(
        r"\d+\s*세|개월|미취학|초등|중학생|고등학생|청소년|성인|어린이|유아",
        text
    ):
        return "연령제한"

    # 전체 대상
    if re.search(
        r"누구나|전체\s*관람가|전\s*연령|모든\s*시민|관심있는\s*누구나",
        text
    ):
        return "전체"

    return "기타"

In [14]:
def classify_target(text):
    if pd.isna(text):
        return "기타"

    text = str(text).strip()

    # 1. 전체
    if re.search(
        r"누구나|전체\s*관람|전\s*연령|^전체$|모든\s*시민|일반시민",
        text
    ):
        return "전체"

    # 2. 고령층 - 성인보다 먼저!
    if re.search(
        r"어르신|노인|시니어|고령|65\s*세\s*이상|만\s*65\s*세\s*이상",
        text
    ):
        return "고령"

    # 3. 명시적 대상
    if re.search(r"유아|미취학|어린이|아동|초등", text):
        return "아동"

    if re.search(r"청소년|중학생|고등학생", text):
        return "청소년"

    if re.search(r"청년|대학생", text):
        return "청년"

    if re.search(r"성인|일반인|직장인", text):
        return "성인"

    # 4. 숫자로 최소 관람연령만 존재
    if re.search(r"(?:만\s*)?\d+\s*세\s*이상", text):
        return "전체"

    if re.search(r"\d+\s*세\s*이하.*(?:불가|금지)", text):
        return "전체"

    # 5. 개월 수로 관람 가능 연령만 제한
    if re.search(r"\d+\s*개월\s*이상", text):
        return "전체"

    if re.search(
        r"가족|부모|학부모|엄마|아빠|양육자|보호자", text):
        return "가족"

    return "기타"

In [15]:
def extract_min_age(text):
    if pd.isna(text):
        return np.nan

    text = str(text)

    # 예: 8세 이상, 만 7세 이상
    match = re.search(r"(?:만\s*)?(\d+)\s*세\s*이상", text)

    if match:
        return int(match.group(1))

    # 예: 7세 이하 입장불가 → 8세 이상
    match = re.search(r"(\d+)\s*세\s*이하.*(?:불가|금지)", text)

    if match:
        return int(match.group(1)) + 1

    return np.nan

In [16]:
df_cultural["audience_scope"] = (
    df_cultural["USE_TRGT"].apply(classify_scope)
)

df_cultural["target_group"] = (
    df_cultural["USE_TRGT"].apply(classify_target)
)

df_cultural["min_age"] = (
    df_cultural["USE_TRGT"].apply(extract_min_age)
)

In [17]:
df_cultural[
    ["USE_TRGT", "audience_scope", "target_group", "min_age"]
].head(20)

,USE_TRGT,audience_scope,target_group,min_age
0,초등학교 특수학급,연령제한,아동,NaN
1,초등학교 특수학급,연령제한,아동,NaN
2,동물원 관람객,기타,기타,NaN
3,초등학교 1~3학년,연령제한,아동,NaN
4,누구나,전체,전체,NaN
5,"초, 중, 고등학교 단체",기타,기타,NaN
6,"청소년(중, 고등학교 특수학급)",연령제한,청소년,NaN
7,"중, 고등학생 및 학부모",연령제한,청소년,NaN
8,초등학교 특수학급,연령제한,아동,NaN
9,"단체, 동호회 / 회차별 10명",기타,기타,NaN


In [18]:
df_cultural["audience_scope"].value_counts()

audience_scope
연령제한       5547
전체         4830
기타         1179
지역제한        191
회원·특정대상     184
Name: count, dtype: int64

In [19]:
df_cultural["target_group"].value_counts()

target_group
전체     7101
아동     2524
기타     1102
성인      738
청소년     255
가족       92
청년       70
고령       49
Name: count, dtype: int64

In [20]:
df_cultural.loc[
    df_cultural["target_group"] == "기타",
    "USE_TRGT"
].value_counts().head(100)

USE_TRGT
프로그램별 상이                          89
서울도서관 회원                          62
일반 30명                            26
2017년생부터 관람 가능                    26
2018년 이전 출생자 관람 가능 (2018년생 포함)    22
                                  ..
관련 분야 취업을 희망하는 구직 여성               2
도서관 이용자                            2
전문공연예술인                            2
추후 공지                              2
행사별 상이                             2
Name: count, Length: 100, dtype: int64

# 인구 분포 그룹화

In [22]:
p_data

,gu_name,total_population,age_0_4,age_5_9,age_10_14,age_15_19,age_20_24,age_25_29,age_30_34,age_35_39,...,age_60_64,age_65_69,age_70_74,age_75_79,age_80_84,age_85_89,age_90_94,age_95_99,age_100_plus,year
0,중구,120544,2562,2754,2803,3080,6346,10431,12014,9582,...,9568,8841,6021,4824,3747,2027,728,198,22,2024
1,성동구,273669,7563,8411,8237,9094,16106,22847,24802,21622,...,20751,17867,11745,9348,7001,3634,1156,245,34,2024
2,금천구,225159,4099,4927,5885,6540,10779,22791,23016,15502,...,19847,17033,11508,8644,5890,2860,899,168,35,2024
3,종로구,138336,2254,3257,4420,4958,8639,12324,11326,8428,...,11293,9723,6478,5389,4337,2500,884,222,41,2024
4,강북구,283597,4189,6365,8573,9832,15996,20206,19307,15295,...,24927,23150,16766,13953,10529,5030,1405,350,44,2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,강동구,499011,14049,17504,22642,21374,20888,33293,40323,36893,...,34819,37085,27574,17866,10515,5157,2024,512,80,2026
71,노원구,481053,10084,12346,19775,23739,28632,31302,32985,28859,...,35723,36537,27187,18828,13211,7659,2932,656,85,2026
72,송파구,650258,16945,22553,29320,28530,28509,46411,56419,50409,...,43897,44554,33458,22131,13080,6680,2696,640,88,2026
73,강남구,552830,12515,18736,31648,31977,25762,35697,41341,37927,...,31848,31511,26418,19031,11649,6028,2272,587,92,2026


In [23]:
p_data["아동_인구"] = (
    p_data["age_0_4"]
    + p_data["age_5_9"]
    + p_data["age_10_14"]
)

p_data["청소년_인구"] = (
    p_data["age_15_19"]
)

p_data["청년_인구"] = (
    p_data["age_20_24"]
    + p_data["age_25_29"]
    + p_data["age_30_34"]
    + p_data["age_35_39"]
)

p_data["성인_인구"] = (
    p_data["age_40_44"]
    + p_data["age_45_49"]
    + p_data["age_50_54"]
    + p_data["age_55_59"]
    + p_data["age_60_64"]
)

p_data["고령_인구"] = (
    p_data["age_65_69"]
    + p_data["age_70_74"]
    + p_data["age_75_79"]
    + p_data["age_80_84"]
    + p_data["age_85_89"]
    + p_data["age_90_94"]
    + p_data["age_95_99"]
    + p_data["age_100_plus"]
)

In [24]:
p_data.head()

,gu_name,total_population,age_0_4,age_5_9,age_10_14,age_15_19,age_20_24,age_25_29,age_30_34,age_35_39,...,age_85_89,age_90_94,age_95_99,age_100_plus,year,아동_인구,청소년_인구,청년_인구,성인_인구,고령_인구
0,중구,120544,2562,2754,2803,3080,6346,10431,12014,9582,...,2027,728,198,22,2024,8119,3080,38373,44564,26408
1,성동구,273669,7563,8411,8237,9094,16106,22847,24802,21622,...,3634,1156,245,34,2024,24211,9094,85377,103957,51030
2,금천구,225159,4099,4927,5885,6540,10779,22791,23016,15502,...,2860,899,168,35,2024,14911,6540,72088,84583,47037
3,종로구,138336,2254,3257,4420,4958,8639,12324,11326,8428,...,2500,884,222,41,2024,9931,4958,40717,53156,29574
4,강북구,283597,4189,6365,8573,9832,15996,20206,19307,15295,...,5030,1405,350,44,2024,19127,9832,70804,112607,71227


In [25]:
p_data.columns

Index(['gu_name', 'total_population', 'age_0_4', 'age_5_9', 'age_10_14',
       'age_15_19', 'age_20_24', 'age_25_29', 'age_30_34', 'age_35_39',
       'age_40_44', 'age_45_49', 'age_50_54', 'age_55_59', 'age_60_64',
       'age_65_69', 'age_70_74', 'age_75_79', 'age_80_84', 'age_85_89',
       'age_90_94', 'age_95_99', 'age_100_plus', 'year', '아동_인구', '청소년_인구',
       '청년_인구', '성인_인구', '고령_인구'],
      dtype='str')

In [27]:
df_cultural.columns

Index(['CODENAME', 'GUNAME', 'TITLE', 'PLACE', 'ORG_NAME', 'USE_TRGT',
       'USE_FEE', 'RGSTDATE', 'TICKET', 'STRTDATE', 'END_DATE', 'THEMECODE',
       'LOT', 'LAT', 'IS_FREE', 'PRO_TIME', 'ORG_LINK', 'HMPG_ADDR',
       'event_year', 'audience_scope', 'target_group', 'min_age'],
      dtype='str')

In [26]:
p_data_c = p_data[['year', 'gu_name', '아동_인구', '청소년_인구',
       '청년_인구', '성인_인구', '고령_인구', 'total_population']]

# 데이터 품질 저하요인 발견

- 시각화 도중 좌표 정보와 자치구가 매칭이 안되고 있는 데이터들 발견. 추가 처리방법 고민

In [66]:
jongno = df_cultural[
    df_cultural["GUNAME"] == "종로구"
].copy()

jongno[[
    "TITLE", "PLACE", "GUNAME", "LAT", "LOT"
]].head()

,TITLE,PLACE,GUNAME,LAT,LOT
588,이선미 피아노 독주회,예술의전당 인춘아트홀,종로구,37.480240,127.014215
1092,[구로문화재단] 연극 [감찰관],구로아트밸리 예술극장,종로구,37.496625,126.890315
1306,[서울아트책보고] 힐링 강연쇼 - 거인의 어깨 2부 [빌드업의 거인 이주호],서울아트책보고 워크숍룸,종로구,37.498213,126.867089
1726,[서울외국인주민센터] 세계문화탐방 [일상 속 세계정거장] 콜롬비아편,광화문역 인근(개별공지),종로구,37.504736,126.897409
1736,[서울외국인주민센터] 세계문화탐방 [일상 속 세계정거장],지하철 3호선 안국역 인근(※추후 개별 안내),종로구,37.504736,126.897409


In [69]:
jongno[[
    "TITLE",
    "PLACE",
    "ORG_NAME",
    "GUNAME",
    "LAT",
    "LOT"
]].sort_values("LAT")

,TITLE,PLACE,ORG_NAME,GUNAME,LAT,LOT
588,이선미 피아노 독주회,예술의전당 인춘아트홀,기타,종로구,37.480240,127.014215
1092,[구로문화재단] 연극 [감찰관],구로아트밸리 예술극장,구로문화재단,종로구,37.496625,126.890315
1306,[서울아트책보고] 힐링 강연쇼 - 거인의 어깨 2부 [빌드업의 거인 이주호],서울아트책보고 워크숍룸,서울아트책보고,종로구,37.498213,126.867089
1726,[서울외국인주민센터] 세계문화탐방 [일상 속 세계정거장] 콜롬비아편,광화문역 인근(개별공지),서울시청,종로구,37.504736,126.897409
1736,[서울외국인주민센터] 세계문화탐방 [일상 속 세계정거장],지하철 3호선 안국역 인근(※추후 개별 안내),서울시청,종로구,37.504736,126.897409
...,...,...,...,...,...,...
10708,Catherine & Renato CASCIANI: Somebody has to c...,서울시 종로구 토탈미술관 전관,기타,종로구,37.612122,126.975675
10709,난지도·메타-복스 40: 녹아내린 모든 견고함,서울시 종로구 토탈미술관 전관,기타,종로구,37.612122,126.975675
10714,2025 제13회 자문밖문화축제 [예술적 삶을 위한 빛 Seoul Art Twili...,"종로구 평창동, 부암동, 홍지동, 신영동, 구기동 일대 (프로그램별 상이)",기타,종로구,37.612245,126.975182
10735,[종로문화재단] 2026 종로 문화예술 지원사업 [디그업 종로] 선정 프로젝트 킴콜...,키미아트 (서울시 종로구 평창30길 47),종로문화재단,종로구,37.613008,126.974293


In [70]:
jongno[[
    "TITLE",
    "PLACE",
    "ORG_NAME",
    "GUNAME",
    "LAT",
    "LOT"
]].sort_values("LOT")

,TITLE,PLACE,ORG_NAME,GUNAME,LAT,LOT
1306,[서울아트책보고] 힐링 강연쇼 - 거인의 어깨 2부 [빌드업의 거인 이주호],서울아트책보고 워크숍룸,서울아트책보고,종로구,37.498213,126.867089
1092,[구로문화재단] 연극 [감찰관],구로아트밸리 예술극장,구로문화재단,종로구,37.496625,126.890315
1736,[서울외국인주민센터] 세계문화탐방 [일상 속 세계정거장],지하철 3호선 안국역 인근(※추후 개별 안내),서울시청,종로구,37.504736,126.897409
1726,[서울외국인주민센터] 세계문화탐방 [일상 속 세계정거장] 콜롬비아편,광화문역 인근(개별공지),서울시청,종로구,37.504736,126.897409
1738,"[서울외국인주민센터] 창신동, 네팔타운 지역탐방",창신동 일대,서울시청,종로구,37.504736,126.897409
...,...,...,...,...,...,...
10505,"[한국예술종합학교] 한국예술연구소 2026년 학술대회 [돌봄의 예술, 담론과 실천의...",한국예술종합학교 대학로 캠퍼스 1층 강당,기타,종로구,37.606358,127.053941
10506,"[한국예술종합학교 한국예술연구소] 2025년 국제학술대회 [구독시대의 예술: 창작,...",한국예술종합학교 대학로 캠퍼스 1층 강당(서울특별시 종로구 창경궁로 215),기타,종로구,37.606358,127.053941
2010,뮤지엄김치간 재개관 10주년 기념 행사 [김치 Remix],뮤지엄김치간,기타,종로구,37.512944,127.057114
2011,뮤지엄김치간 재개관 9주년 기념 행사 [뮤지엄김치간 Story & History],뮤지엄김치간,기타,종로구,37.512944,127.057114


In [77]:
boundary = gpd.read_file(
    "..//data//tb_etl_tp_admstr_zone_lgldong_bndry//admstr_zone_lgldong_bndry_24.shp",
    encoding="cp949"
)

boundary.head()

,EMD_CD,COL_ADM_SE,EMD_NM,SGG_OID,geometry
0,11110101,11110,청운동,1036,"POLYGON ((196593.774 554114.361, 196611.342 55..."
1,11110102,11110,신교동,307,"POLYGON ((196912.513 554003.597, 196920.988 55..."
2,11110103,11110,궁정동,1034,"POLYGON ((197377.934 553847.046, 197388.359 55..."
3,11110104,11110,효자동,309,"POLYGON ((197664.978 553742.317, 197663.825 55..."
4,11110105,11110,창성동,316,"POLYGON ((197662.098 553585.836, 197662.501 55..."


In [78]:
boundary.columns

Index(['EMD_CD', 'COL_ADM_SE', 'EMD_NM', 'SGG_OID', 'geometry'], dtype='str')

In [79]:
boundary.crs

<Projected CRS: EPSG:5186>
Name: KGD2002 / Central Belt 2010
Axis Info [cartesian]:
- X[north]: Northing (metre)
- Y[east]: Easting (metre)
Area of Use:
- name: Republic of Korea (South Korea) - onshore between 126°E and 128°E.
- bounds: (126.0, 33.14, 128.0, 38.33)
Coordinate Operation:
- name: Korea Central Belt 2010
- method: Transverse Mercator
Datum: Korean Geodetic Datum 2002
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

In [80]:
boundary.shape

(467, 5)

In [81]:
gu_boundary = (
    boundary
    .dissolve(by="COL_ADM_SE")
    .reset_index()
)

In [82]:
gu_boundary.shape

(25, 5)

In [83]:
gu_boundary[["COL_ADM_SE", "geometry"]].head()

,COL_ADM_SE,geometry
0,11110,"MULTIPOLYGON (((197219.852 552015.04, 197214.8..."
1,11140,"POLYGON ((198676.972 550436.315, 198656.186 55..."
2,11170,"POLYGON ((198319.723 545229.009, 198319.608 54..."
3,11200,"POLYGON ((200762.183 549595.745, 200769.633 54..."
4,11215,"POLYGON ((207845.426 547427.84, 207802.277 547..."


In [84]:
events_gdf = gpd.GeoDataFrame(
    df_cultural.copy(),
    geometry=gpd.points_from_xy(
        df_cultural["LOT"],
        df_cultural["LAT"]
    ),
    crs="EPSG:4326"
)

In [85]:
gu_boundary = gu_boundary.to_crs("EPSG:4326")

In [86]:
print(events_gdf.crs)
print(gu_boundary.crs)

EPSG:4326
EPSG:4326


In [87]:
gu_map = gu_boundary[
    ["COL_ADM_SE", "geometry"]
].copy()

matched = gpd.sjoin(
    events_gdf,
    gu_map,
    how="left",
    predicate="within"
)

matched.head()

,CODENAME,GUNAME,TITLE,PLACE,ORG_NAME,USE_TRGT,USE_FEE,RGSTDATE,TICKET,STRTDATE,...,PRO_TIME,ORG_LINK,HMPG_ADDR,event_year,audience_scope,target_group,min_age,geometry,index_right,COL_ADM_SE
0,교육/체험,광진구,[서울대공원] 2025 띵똥! 서울동물원 교육꾸러미 비대면 교육 (초등학교 특수학급...,각 기관,서울대공원,초등학교 특수학급,NaN,2025-03-11,기관,2025-04-01,...,각 기관 별 상이,https://grandpark.seoul.go.kr/edu/program/view...,https://culture.seoul.go.kr/culture/culture/cu...,2025,연령제한,아동,NaN,POINT (127.0141 37.43643),NaN,NaN
1,교육/체험,광진구,[서울대공원] 2025 [학교에서 만나는 동물원] 비대면 교육,각 기관,서울대공원,초등학교 특수학급,NaN,2025-03-11,기관,2025-04-01,...,각 기관 별 상이,https://grandpark.seoul.go.kr/edu/program/view...,https://culture.seoul.go.kr/culture/culture/cu...,2025,연령제한,아동,NaN,POINT (127.0141 37.43643),NaN,NaN
2,교육/체험,광진구,[서울대공원] 2024 생물다양성의 날 기념 멸종위기종 보전 교육,동물원 내 100주년 기념 광장,서울대공원,동물원 관람객,홈페이지 참고,2024-05-21,기관,2024-05-22,...,"10:30~11:30, 13:20~15:30",https://grandpark.seoul.go.kr/board/view/ko/S0...,https://culture.seoul.go.kr/culture/culture/cu...,2024,기타,기타,NaN,POINT (127.0141 37.43643),NaN,NaN
3,교육/체험,광진구,[서울대공원] 사자야! 신나게 놀아보자!,"동물원 내 식물원(세미나실), 사자 실내 관람장",서울대공원,초등학교 1~3학년,NaN,2024-07-23,기관,2024-07-31,...,13:40-14:40,https://grandpark.seoul.go.kr/edu/program/view...,https://culture.seoul.go.kr/culture/culture/cu...,2024,연령제한,아동,NaN,POINT (127.0141 37.43643),NaN,NaN
4,축제-자연/경관,광진구,[서울대공원] 2026년 봄꽃축제,서울대공원,서울대공원,누구나,입장료 별도,2026-03-24,기관,2026-04-04,...,9:00 ~ 18:00,https://grandpark.seoul.go.kr/munhwa/munhwaVie...,https://culture.seoul.go.kr/culture/culture/cu...,2026,전체,전체,NaN,POINT (127.0141 37.43643),NaN,NaN


In [88]:
matched.isna().sum()

CODENAME              0
GUNAME                0
TITLE                 0
PLACE                 0
ORG_NAME              0
USE_TRGT              0
USE_FEE            6631
RGSTDATE              0
TICKET                0
STRTDATE              0
END_DATE              0
THEMECODE            65
LOT                   0
LAT                   0
IS_FREE               0
PRO_TIME              0
ORG_LINK              0
HMPG_ADDR             0
event_year            0
audience_scope        0
target_group          0
min_age           10134
geometry              0
index_right          64
COL_ADM_SE           64
dtype: int64

In [89]:
unmatched = matched[
    matched["COL_ADM_SE"].isna()
].copy()

unmatched[
    ["TITLE", "PLACE", "GUNAME", "LAT", "LOT"]
]

,TITLE,PLACE,GUNAME,LAT,LOT
0,[서울대공원] 2025 띵똥! 서울동물원 교육꾸러미 비대면 교육 (초등학교 특수학급...,각 기관,광진구,37.436431,127.014098
1,[서울대공원] 2025 [학교에서 만나는 동물원] 비대면 교육,각 기관,광진구,37.436431,127.014098
2,[서울대공원] 2024 생물다양성의 날 기념 멸종위기종 보전 교육,동물원 내 100주년 기념 광장,광진구,37.436431,127.014098
3,[서울대공원] 사자야! 신나게 놀아보자!,"동물원 내 식물원(세미나실), 사자 실내 관람장",광진구,37.436431,127.014098
4,[서울대공원] 2026년 봄꽃축제,서울대공원,광진구,37.436431,127.014098
...,...,...,...,...,...
59,[서울대공원] 물범 친구들을 소개합니다,서울동물원 해양관,광진구,37.436431,127.014098
60,[서울대공원] 출동! 어린이 동물 탐험대(비대면/수어통역영상),온라인,광진구,37.436431,127.014098
61,[서울대공원] 띵똥! 동물원 교육꾸러미 비대면 교육,온라인,광진구,37.436431,127.014098
62,[광진문화재단] 지역문화 활성화 사업 [2024 나루랑 버스킹],"청춘뜨락, 아차산 토요한마당, 아차산 어울림광장, 서울어린이대공원 후문",광진구,37.436431,127.014098


In [ ]:
unmatched["GUNAME"].value_counts()

GUNAME
광진구    64
Name: count, dtype: int64

In [91]:
unmatched["ORG_NAME"].value_counts()

ORG_NAME
서울대공원     63
광진문화재단     1
Name: count, dtype: int64

In [92]:
unmatched[["ORG_NAME", "PLACE"]].drop_duplicates()

,ORG_NAME,PLACE
0,서울대공원,각 기관
2,서울대공원,동물원 내 100주년 기념 광장
3,서울대공원,"동물원 내 식물원(세미나실), 사자 실내 관람장"
4,서울대공원,서울대공원
9,서울대공원,서울대공원 내 산림치유센터
10,서울대공원,서울대공원 동물원 일대
11,서울대공원,서울대공원 동물원 제1아프리카관
12,서울대공원,서울대공원 동양관 실내
13,서울대공원,서울대공원 만남의 광장 인근 잔디밭
14,서울대공원,서울대공원 만남의 광장 일대


In [93]:
matched["analysis_gu"] = matched["COL_ADM_SE"]

In [94]:
matched["location_status"] = "서울 내"

matched.loc[
    matched["COL_ADM_SE"].isna() &
    (matched["ORG_NAME"] == "서울대공원"),
    "location_status"
] = "서울 외"

In [95]:
matched[
    (matched["COL_ADM_SE"].isna()) &
    (matched["ORG_NAME"] == "광진문화재단")
][["TITLE", "PLACE", "LAT", "LOT"]]

,TITLE,PLACE,LAT,LOT
62,[광진문화재단] 지역문화 활성화 사업 [2024 나루랑 버스킹],"청춘뜨락, 아차산 토요한마당, 아차산 어울림광장, 서울어린이대공원 후문",37.436431,127.014098


In [97]:
# 1. 먼저 좌표 기반 자치구 코드 저장
matched["analysis_gu"] = matched["COL_ADM_SE"]

# 2. 그 다음 광진문화재단 1건 수동 보정
mask_gwangjin_error = (
    matched["COL_ADM_SE"].isna()
    & (matched["ORG_NAME"] == "광진문화재단")
)

matched.loc[mask_gwangjin_error, "analysis_gu"] = "광진구"
matched.loc[mask_gwangjin_error, "location_status"] = "좌표오류_수동보정"

In [98]:
gu_code_map = {
    "11110": "종로구",
    "11140": "중구",
    "11170": "용산구",
    "11200": "성동구",
    "11215": "광진구",
    "11230": "동대문구",
    "11260": "중랑구",
    "11290": "성북구",
    "11305": "강북구",
    "11320": "도봉구",
    "11350": "노원구",
    "11380": "은평구",
    "11410": "서대문구",
    "11440": "마포구",
    "11470": "양천구",
    "11500": "강서구",
    "11530": "구로구",
    "11545": "금천구",
    "11560": "영등포구",
    "11590": "동작구",
    "11620": "관악구",
    "11650": "서초구",
    "11680": "강남구",
    "11710": "송파구",
    "11740": "강동구"
}

# 2. 좌표 기준 실제 자치구명 생성
matched["analysis_gu"] = (
    matched["COL_ADM_SE"]
    .astype("Int64")
    .astype("string")
    .map(gu_code_map)
)

matched["location_status"] = "정상"

mask_seoulpark = (
    matched["COL_ADM_SE"].isna()
    & (matched["ORG_NAME"] == "서울대공원")
)

matched.loc[
    mask_seoulpark,
    "location_status"
] = "서울외"

mask_gwangjin_error = (
    matched["COL_ADM_SE"].isna()
    & (matched["ORG_NAME"] == "광진문화재단")
)

matched.loc[
    mask_gwangjin_error,
    "analysis_gu"
] = "광진구"

matched.loc[
    mask_gwangjin_error,
    "location_status"
] = "좌표오류_수동보정"

In [100]:
matched["analysis_gu"].isna().sum()

np.int64(63)

In [101]:
seoul_events = matched[
    matched["analysis_gu"].notna()
].copy()

In [102]:
seoul_events["gu_match"] = (
    seoul_events["GUNAME"] == seoul_events["analysis_gu"]
)

In [103]:
seoul_events["gu_match"].value_counts()

gu_match
True     11649
False      219
Name: count, dtype: int64

In [104]:
seoul_events["gu_match"].value_counts(
    normalize=True
) * 100

gu_match
True     98.154702
False     1.845298
Name: proportion, dtype: float64

In [105]:
gu_mismatch = seoul_events[
    seoul_events["GUNAME"] != seoul_events["analysis_gu"]
].copy()

In [106]:
gu_mismatch[
    [
        "TITLE",
        "PLACE",
        "ORG_NAME",
        "GUNAME",
        "analysis_gu",
        "LAT",
        "LOT"
    ]
]

,TITLE,PLACE,ORG_NAME,GUNAME,analysis_gu,LAT,LOT
72,"[금천문화재단] 2024 N개의 금천 [금천피플 : 지역, 사람, 삶을 기록하기]",만천명월 예술인家 2층,금천문화재단,강남구,금천구,37.450698,126.907445
190,[관악문화재단] 2026 관악아트홀 신년맞이 [필 하모닉 앙상블],관악아트홀,관악문화재단,광진구,관악구,37.467553,126.944962
275,2024 서울 일러스트코리아 winter,양재 aT센터,기타,강남구,서초구,37.468279,127.039136
352,"[서울시립 남서울미술관] 전국광: 쌓는 친구, 허무는 친구","서울시립 남서울미술관 1층 야외, 2층 전시실",서울시립미술관,광진구,관악구,37.476054,126.979466
458,[서울시서남권직장맘지원센터] 출산휴가·육아휴직 서식작성법 교육,온라인 교육(ZOOM),기타,금천구,구로구,37.479848,126.895318
...,...,...,...,...,...,...,...
11838,[서울로봇인공지능과학관] 강연/라임 콜로키엄 [2026년 제4회_피지컬 AI 및 자...,서울로봇인공지능과학관 1층 로비,서울시립과학관,강북구,도봉구,37.655584,127.049506
11853,[도봉문화재단] 2024 김수영문학관 상반기 인문학 강연,김수영문학관 4층 강당,도봉문화재단,노원구,도봉구,37.659924,127.023802
11872,[도봉문화재단] 도봉구청 잔디마당 문화체육프로그램(힐링요가&치유 스트레칭체조),도봉구청 앞 잔디마당,도봉문화재단,노원구,도봉구,37.668716,127.047131
11890,도봉 마을아카이브 연계 민주주의 · 인권 기획전 《기억×기록》전시연계워크숍 [정말 ...,"김근태기념도서관 2층 공간, 마루",도봉구청,은평구,도봉구,37.687667,127.044020


In [108]:
gu_mismatch.groupby(
    ["GUNAME", "analysis_gu"]
).size().sort_values(ascending=False).head(30)

GUNAME  analysis_gu
종로구     중구             38
은평구     종로구            37
중구      종로구            13
동대문구    중구              6
강남구     서초구             6
강북구     종로구             5
중랑구     종로구             5
강남구     종로구             4
동대문구    도봉구             4
양천구     영등포구            4
노원구     도봉구             3
종로구     영등포구            3
은평구     서대문구            3
영등포구    용산구             3
동작구     마포구             3
마포구     서대문구            3
중구      영등포구            3
광진구     관악구             3
서대문구    서초구             2
송파구     양천구             2
양천구     송파구             2
관악구     동작구             2
금천구     노원구             2
서초구     성동구             2
강서구     종로구             2
양천구     종로구             2
종로구     강남구             2
        성북구             2
        구로구             2
금천구     구로구             2
dtype: int64

# <>

# 데이터 마트 생성

## 집계 데이터 마트

In [111]:
seoul_events.head(1)

,CODENAME,GUNAME,TITLE,PLACE,ORG_NAME,USE_TRGT,USE_FEE,RGSTDATE,TICKET,STRTDATE,...,event_year,audience_scope,target_group,min_age,geometry,index_right,COL_ADM_SE,analysis_gu,location_status,gu_match
62,콘서트,광진구,[광진문화재단] 지역문화 활성화 사업 [2024 나루랑 버스킹],"청춘뜨락, 아차산 토요한마당, 아차산 어울림광장, 서울어린이대공원 후문",광진문화재단,전체관람가,NaN,2024-03-22,기관,2024-04-05,...,2024,전체,전체,NaN,POINT (127.0141 37.43643),NaN,NaN,광진구,좌표오류_수동보정,True


In [112]:
c_df = seoul_events.groupby(['event_year', 'analysis_gu', 'target_group'])['TITLE'].count().reset_index()
c_df = pd.pivot_table(c_df, index=['analysis_gu', 'event_year'], columns=['target_group'], values='TITLE')
c_df = c_df.reset_index()
c_df = c_df.rename(columns={
    '가족':'family_c',
    '고령':'old_age',
    '기타':'Other',
    '성인':'midlife',
    '아동':'kiz',
    '전체':'all',
    '청년':'adult',
    '청소년':'youth',
    
})
c_df.shape

(75, 10)

In [113]:
p_data_c.shape

(75, 8)

In [115]:
mart = pd.merge(c_df, p_data_c, left_on=['event_year', 'analysis_gu'], right_on=['year', 'gu_name'], how='left')
mart = mart.fillna(0)

target_cols = [
    "family_c", "old_age", "Other", "midlife",
    "kiz", "all", "adult", "youth"
]


mart["total_cul"] = mart[target_cols].sum(axis=1)
mart = mart.drop(columns=['year', 'gu_name'])

mart["total_Level"] = mart["total_cul"] / mart["total_population"] * 10000
mart["kiz_Level"] = mart["kiz"] / mart["아동_인구"] * 10000
mart["youth_Level"] = mart["youth"] / mart["청소년_인구"] * 10000
mart["adult_Level"] = mart["adult"] / mart["청년_인구"] * 10000
mart["midlife_Level"] = mart["midlife"] / mart["성인_인구"] * 10000
mart["old_age_Level"] = mart["old_age"] / mart["고령_인구"] * 10000
mart.head()

,analysis_gu,event_year,family_c,old_age,Other,midlife,kiz,all,adult,youth,...,성인_인구,고령_인구,total_population,total_cul,total_Level,kiz_Level,youth_Level,adult_Level,midlife_Level,old_age_Level
0,강남구,2024,0.0,0.0,5.0,22.0,50.0,131.0,0.0,0.0,...,221772,91291,557345,208.0,3.731979,7.627067,0.0,0.0,0.99201,0.0
1,강남구,2025,1.0,0.0,2.0,1.0,11.0,85.0,0.0,1.0,...,221115,95349,556330,101.0,1.815469,1.713502,0.313913,0.0,0.045225,0.0
2,강남구,2026,1.0,0.0,3.0,5.0,7.0,65.0,0.0,0.0,...,219639,97588,552830,81.0,1.465188,1.112895,0.0,0.0,0.227646,0.0
3,강동구,2024,0.0,0.0,6.0,7.0,8.0,57.0,0.0,4.0,...,186207,89687,476942,82.0,1.719287,1.538107,2.01096,0.0,0.375926,0.0
4,강동구,2025,1.0,0.0,5.0,5.0,3.0,27.0,0.0,1.0,...,192376,97907,499127,42.0,0.841469,0.55058,0.469704,0.0,0.259908,0.0


In [116]:
mart.columns

Index(['analysis_gu', 'event_year', 'family_c', 'old_age', 'Other', 'midlife',
       'kiz', 'all', 'adult', 'youth', '아동_인구', '청소년_인구', '청년_인구', '성인_인구',
       '고령_인구', 'total_population', 'total_cul', 'total_Level', 'kiz_Level',
       'youth_Level', 'adult_Level', 'midlife_Level', 'old_age_Level'],
      dtype='str')

In [117]:
mart.head()

,analysis_gu,event_year,family_c,old_age,Other,midlife,kiz,all,adult,youth,...,성인_인구,고령_인구,total_population,total_cul,total_Level,kiz_Level,youth_Level,adult_Level,midlife_Level,old_age_Level
0,강남구,2024,0.0,0.0,5.0,22.0,50.0,131.0,0.0,0.0,...,221772,91291,557345,208.0,3.731979,7.627067,0.0,0.0,0.99201,0.0
1,강남구,2025,1.0,0.0,2.0,1.0,11.0,85.0,0.0,1.0,...,221115,95349,556330,101.0,1.815469,1.713502,0.313913,0.0,0.045225,0.0
2,강남구,2026,1.0,0.0,3.0,5.0,7.0,65.0,0.0,0.0,...,219639,97588,552830,81.0,1.465188,1.112895,0.0,0.0,0.227646,0.0
3,강동구,2024,0.0,0.0,6.0,7.0,8.0,57.0,0.0,4.0,...,186207,89687,476942,82.0,1.719287,1.538107,2.01096,0.0,0.375926,0.0
4,강동구,2025,1.0,0.0,5.0,5.0,3.0,27.0,0.0,1.0,...,192376,97907,499127,42.0,0.841469,0.55058,0.469704,0.0,0.259908,0.0


# bigquery 업로드

In [119]:
seoul_events_bq = seoul_events.drop(
    columns=[
        "geometry",
        "index_right",
        "COL_ADM_SE",
        "gu_match"
    ],
    errors="ignore"
).copy()

In [120]:
table_id = "sprint-mission17.mission17_kkw.cultural_mart_v2"
table_id1 = "sprint-mission17.mission17_kkw.agg_mart_v2"

job = client.load_table_from_dataframe(
    seoul_events_bq,
    table_id
)

job.result()

job1 = client.load_table_from_dataframe(
    mart,
    table_id1
)

job1.result()

print("업로드 완료!")

c:\workspace\sprint_mission17\.venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(
c:\workspace\sprint_mission17\.venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


업로드 완료!
